# Vehicle Detection Thesis — Dataset Exploration

This notebook covers:
1. Dataset statistics (class distribution, image sizes, annotation density)
2. Visual samples from KITTI, BDD100K, UA-DETRAC
3. Cross-dataset class overlap analysis
4. Sanity check of loaders

In [ ]:
import sys
sys.path.insert(0, '..')   # project root

import numpy as np
import matplotlib.pyplot as plt
import torch
from collections import Counter
from omegaconf import OmegaConf

# Load base config
cfg = OmegaConf.load('../configs/base.yaml')
CLASSES = list(cfg.data.classes)
print('Classes:', CLASSES)

## 1. KITTI

In [ ]:
from data.kitti.kitti_dataset import KITTIDataset

kitti_train = KITTIDataset(
    root='../data/kitti',
    split='train',
    classes_filter=CLASSES,
)
kitti_val = KITTIDataset(
    root='../data/kitti',
    split='val',
    classes_filter=CLASSES,
)
print(kitti_train)
print(kitti_val)

In [ ]:
# Class distribution
def count_classes(dataset, class_names):
    counts = Counter()
    n_boxes_per_image = []
    for i in range(len(dataset)):
        item = dataset[i]
        labels = item['labels'].tolist()
        for l in labels:
            counts[class_names[l]] += 1
        n_boxes_per_image.append(len(labels))
    return counts, n_boxes_per_image

kitti_counts, kitti_density = count_classes(kitti_train, CLASSES)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(kitti_counts.keys(), kitti_counts.values(), color='steelblue')
axes[0].set_title('KITTI — Class distribution (train)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

axes[1].hist(kitti_density, bins=20, color='steelblue', edgecolor='black')
axes[1].set_title('KITTI — Objects per image')
axes[1].set_xlabel('# objects')
axes[1].set_ylabel('# images')
plt.tight_layout()
plt.show()

In [ ]:
# Visual samples
from utils.visualization import draw_detections
import cv2

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for i, ax in enumerate(axes.flatten()):
    item = kitti_train[i * 50]
    img = (item['image'].permute(1,2,0).numpy() * 255).astype(np.uint8)
    # Denormalize boxes
    h, w = img.shape[:2]
    boxes = item['boxes'].clone()
    boxes[:, [0,2]] *= w
    boxes[:, [1,3]] *= h
    annotated = draw_detections(img, boxes, item['labels'],
                                class_names=CLASSES, is_bgr=False)
    ax.imshow(annotated)
    ax.set_title(f"KITTI — {item['image_id']}")
    ax.axis('off')
plt.suptitle('KITTI Training Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. BDD100K

In [ ]:
from data.bdd100k.bdd100k_dataset import BDD100KDataset

bdd_train = BDD100KDataset(
    root='../data/bdd100k',
    split='train',
    classes_filter=CLASSES,
)
bdd_val = BDD100KDataset(
    root='../data/bdd100k',
    split='val',
    classes_filter=CLASSES,
)
print(bdd_train)
print(bdd_val)

In [ ]:
# Subsample for speed (BDD100K is large)
SAMPLE = 2000
idx = np.random.choice(len(bdd_train), SAMPLE, replace=False)
bdd_sub = torch.utils.data.Subset(bdd_train, idx)

bdd_counts, bdd_density = count_classes(bdd_sub, CLASSES)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(bdd_counts.keys(), bdd_counts.values(), color='coral')
axes[0].set_title(f'BDD100K — Class distribution (sample n={SAMPLE})')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

axes[1].hist(bdd_density, bins=20, color='coral', edgecolor='black')
axes[1].set_title('BDD100K — Objects per image')
axes[1].set_xlabel('# objects')
plt.tight_layout()
plt.show()

## 3. Cross-dataset comparison

In [ ]:
# Side-by-side bar chart
all_counts = {
    'KITTI':   kitti_counts,
    'BDD100K': bdd_counts,
}
datasets = list(all_counts.keys())
x = np.arange(len(CLASSES))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['steelblue', 'coral']
for i, (ds_name, counts) in enumerate(all_counts.items()):
    vals = [counts.get(c, 0) for c in CLASSES]
    # Normalize to percentage
    total = sum(vals)
    vals = [v / total * 100 for v in vals]
    ax.bar(x + i * width, vals, width, label=ds_name, color=colors[i],
           edgecolor='black', linewidth=0.5)

ax.set_xlabel('Class')
ax.set_ylabel('% of annotations')
ax.set_title('Class Distribution Comparison (normalized)')
ax.set_xticks(x + width / 2)
ax.set_xticklabels(CLASSES, rotation=15)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../notebooks/class_distribution_comparison.png', dpi=150)
plt.show()
print('Saved figure.')

## 4. DataLoader sanity check

In [ ]:
from torch.utils.data import DataLoader
from data.kitti.kitti_dataset import collate_fn
from data.transforms import build_transforms

cfg_stub = OmegaConf.create({
    'data': {'img_size': 640, 'classes': CLASSES, 'batch_size': 4, 'num_workers': 0}
})

kitti_loader = DataLoader(
    KITTIDataset(root='../data/kitti', split='val',
                 transforms=build_transforms('val', cfg_stub),
                 classes_filter=CLASSES),
    batch_size=4, collate_fn=collate_fn
)

images, targets, ids, sizes = next(iter(kitti_loader))
print('Batch images shape:', images.shape)
print('Target boxes[0]:', targets[0]['boxes'])
print('Image IDs:', ids)
print('✓ DataLoader OK')